### Examination des résultats avec toutes les combinaisons de coupes possibles

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

In [2]:
tab_fp_without_last_layer =pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_04_29_15h10_55s_SDPu-FP-without-LAST-LAYER/results.csv")
tab_fp_with_last_layer =pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_04_29_15h10_06s_SDPu-FP-with-LAST-LAYER/results.csv")
tab_pp_without_last_layer =pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_04_29_15h08_57s_SDPu-PP-without-LAST-LAYER/results.csv")
tab_pp_with_last_layer =pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_04_29_15h08_43s_SDPu-PP-with-LAST-LAYER/results.csv")    

In [ ]:
tab = pd.concat([tab_fp_without_last_layer, tab_fp_with_last_layer, tab_pp_without_last_layer, tab_pp_with_last_layer], ignore_index=True)

In [3]:
tab = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/6x100-0.026/2026_04_30_11h03_54s_SDPu-FP-all-combinaisons-cuts/results.csv")

In [4]:
tab['data_index'].nunique()

12

In [ ]:
tab.columns


In [ ]:
parameters = ['LAST_LAYER', 'MATRIX_BY_LAYERS', 'USE_STABLE_ACTIVES', 'USE_STABLE_INACTIVES', 'McCormick_beta_z',  'RLT', 'triangularization', 'beta_logits_comparaison', 'beta_logits_comparaison_big_M', 'sum_beta_logits_equal_logit']

In [ ]:
tab['status'].value_counts()

In [ ]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay


optimality = (tab['status'] == 'SolutionStatus.Optimal') | (tab['status'] == 1)
# Régression : uniquement les runs qui ont abouti (optimal_value définie)
tab_optimal = tab[optimality].copy()
X = tab_optimal[parameters].astype(int)
y = tab_optimal['optimal_value']

print(f"Runs optimaux : {len(tab_optimal)} / {len(tab)}")
print(f"Runs non-optimaux (Undefined) : {(~optimality).sum()}")

# Classification : tous les runs
X_clf = tab[parameters].astype(int)
y_clf = (tab['status'] != 'SolutionStatus.Optimal').astype(int)  # 1 = infaisable/indéfini

In [ ]:
tab_non_optimal = tab[~optimality].copy()
tab_non_optimal.value_counts('data_index')

## 1. Analyse groupby — effet marginal de chaque paramètre

Pour chaque paramètre booléen, on compare la moyenne de `optimal_value` quand il est activé vs désactivé.
C'est le diagnostic le plus rapide avant d'aller vers un modèle.

In [ ]:
rows = []
for param in parameters:
    group = tab_optimal.groupby(param)['optimal_value']
    for val, mean in group.mean().items():
        rows.append({'parameter': param, 'value': bool(val), 'mean_optimal_value': mean, 'count': group.count()[val]})

groupby_df = pd.DataFrame(rows)

pivot = groupby_df.pivot(index='parameter', columns='value', values='mean_optimal_value')
pivot.columns = ['disabled', 'enabled']
pivot['delta (enabled - disabled)'] = pivot['enabled'] - pivot['disabled']
pivot.sort_values('delta (enabled - disabled)', ascending=False).round(4)

In [ ]:
only_parameter = 'sum_beta_logits_equal_logit'
mask_only_parameter = (tab_optimal[only_parameter]) & (~tab[parameter] for parameter in parameters if parameter != only_parameter)
tab[mask_only_parameter]

## 2. Decision Tree Regressor

L'arbre capture les **interactions** entre paramètres (ex : "RLT n'aide que si triangularization est aussi activé").
On affiche l'arbre en texte + les feature importances.

In [ ]:
dt = DecisionTreeRegressor(max_depth=4, min_samples_leaf=2, random_state=0)
dt.fit(X, y)

print(f"R² in-sample : {dt.score(X, y):.3f}")
cv_scores = cross_val_score(dt, X, y, cv=5, scoring='r2')
print(f"R² cross-val (5-fold) : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print()
print(export_text(dt, feature_names=parameters))

In [ ]:
fi_dt = pd.Series(dt.feature_importances_, index=parameters).sort_values(ascending=True)
fi_dt.plot.barh(title="Feature importances — Decision Tree Regressor")
plt.xlabel("Importance (réduction de MSE normalisée)")
plt.tight_layout()
plt.show()

## 3. Random Forest Regressor + Permutation Importance

Le Random Forest donne des importances plus stables (moyenne sur 100 arbres).
La **permutation importance** est encore plus fiable : elle mesure la dégradation réelle du R² quand on permute aléatoirement une feature.

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=0)
rf.fit(X, y)

print(f"R² in-sample : {rf.score(X, y):.3f}")
cv_scores_rf = cross_val_score(rf, X, y, cv=5, scoring='r2')
print(f"R² cross-val (5-fold) : {cv_scores_rf.mean():.3f} ± {cv_scores_rf.std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Impurity-based importance (RF) ---
fi_rf = pd.Series(rf.feature_importances_, index=parameters).sort_values(ascending=True)
fi_rf.plot.barh(ax=axes[0], title="Impurity-based importance (Random Forest)")
axes[0].set_xlabel("Importance (réduction de MSE normalisée)")

# --- Permutation importance ---
perm = permutation_importance(rf, X, y, n_repeats=30, random_state=0, scoring='r2')
fi_perm = pd.Series(perm.importances_mean, index=parameters).sort_values(ascending=True)
fi_perm.plot.barh(ax=axes[1], title="Permutation importance (Random Forest)")
axes[1].set_xlabel("Dégradation moyenne du R²")
# Barres d'erreur : écart-type sur les 30 permutations
axes[1].barh(
    range(len(fi_perm)),
    fi_perm.values,
    xerr=perm.importances_std[fi_perm.index.map(list(parameters).index)],
    align='center',
)
axes[1].set_yticks(range(len(fi_perm)))
axes[1].set_yticklabels(fi_perm.index)

plt.tight_layout()
plt.show()

## Analyse de l'infaisabilité

### 4a. Distribution et taux d'infaisabilité par paramètre

Pour chaque paramètre, on regarde le **taux de `SolutionStatus.Undefined`** selon qu'il est activé ou non.
Un taux élevé quand le paramètre est activé signale que cette coupe/configuration cause des problèmes numériques.

In [ ]:
print(f"Distribution globale :\n{tab['status'].value_counts()}\n")
print(f"Taux d'infaisabilité global : {y_clf.mean():.1%}\n")

rows_clf = []
for param in parameters:
    rates = tab.groupby(param).apply(lambda g: (g['status'] != 'SolutionStatus.Optimal').mean())
    counts = tab.groupby(param).size()
    for val, rate in rates.items():
        rows_clf.append({'parameter': param, 'value': bool(val), 'infeasibility_rate': rate, 'count': counts[val]})

clf_df = pd.DataFrame(rows_clf)
pivot_clf = clf_df.pivot(index='parameter', columns='value', values='infeasibility_rate')
pivot_clf.columns = ['disabled', 'enabled']
pivot_clf['delta (enabled - disabled)'] = pivot_clf['enabled'] - pivot_clf['disabled']
pivot_clf.sort_values('delta (enabled - disabled)', ascending=False).round(3)

### 4b. Decision Tree Classifier

L'arbre montre **quelles combinaisons de paramètres mènent à l'infaisabilité**.
`class_weight='balanced'` compense le déséquilibre des classes (peu de cas Undefined vs beaucoup d'Optimal).

In [ ]:
dt_clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, class_weight='balanced', random_state=0)
dt_clf.fit(X_clf, y_clf)

y_pred = dt_clf.predict(X_clf)
print("=== Classification report (in-sample) ===")
print(classification_report(y_clf, y_pred, target_names=['Optimal', 'Infaisable']))

cv_acc = cross_val_score(dt_clf, X_clf, y_clf, cv=5, scoring='balanced_accuracy')
print(f"Balanced accuracy cross-val (5-fold) : {cv_acc.mean():.3f} ± {cv_acc.std():.3f}")

print("\n=== Arbre de décision ===")
print(export_text(dt_clf, feature_names=parameters))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
ConfusionMatrixDisplay.from_predictions(
    y_clf, y_pred, display_labels=['Optimal', 'Infaisable'], ax=axes[0], colorbar=False
)
axes[0].set_title("Matrice de confusion — Decision Tree")

# Feature importances
fi_clf = pd.Series(dt_clf.feature_importances_, index=parameters).sort_values(ascending=True)
fi_clf.plot.barh(ax=axes[1], title="Feature importances — Infaisabilité (Decision Tree)")
axes[1].set_xlabel("Importance (réduction de Gini)")

plt.tight_layout()
plt.show()

### 4c. Random Forest Classifier + Permutation Importance

Pour des importances plus stables. La **balanced accuracy** est la bonne métrique ici (classes déséquilibrées : ~10% d'infaisables).

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=0)
rf_clf.fit(X_clf, y_clf)

cv_rf_clf = cross_val_score(rf_clf, X_clf, y_clf, cv=5, scoring='balanced_accuracy')
print(f"Balanced accuracy cross-val (5-fold) : {cv_rf_clf.mean():.3f} ± {cv_rf_clf.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Impurity-based importance
fi_rf_clf = pd.Series(rf_clf.feature_importances_, index=parameters).sort_values(ascending=True)
fi_rf_clf.plot.barh(ax=axes[0], title="Impurity-based importance (RF Classifier)")
axes[0].set_xlabel("Importance (réduction de Gini)")

# Permutation importance
perm_clf = permutation_importance(rf_clf, X_clf, y_clf, n_repeats=30, random_state=0, scoring='balanced_accuracy')
fi_perm_clf = pd.Series(perm_clf.importances_mean, index=parameters).sort_values(ascending=True)
order = [list(parameters).index(p) for p in fi_perm_clf.index]
axes[1].barh(
    range(len(fi_perm_clf)),
    fi_perm_clf.values,
    xerr=perm_clf.importances_std[order],
    align='center',
)
axes[1].set_yticks(range(len(fi_perm_clf)))
axes[1].set_yticklabels(fi_perm_clf.index)
axes[1].set_title("Permutation importance (RF Classifier)")
axes[1].set_xlabel("Dégradation moyenne de la balanced accuracy")
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()